In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Настройки отображения
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Загрузка данных
df = pd.read_csv('data.csv', encoding='unicode_escape')  # иногда нужен latin-1

# Просмотр
print("Размер данных:", df.shape)
print("\nПервые 5 строк:")
display(df.head())
print("\nИнформация о колонках и типах:")
df.info()
print("\nСтатистика по числовым:")
display(df.describe())

Размер данных: (541909, 8)

Первые 5 строк:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom



Информация о колонках и типах:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

Статистика по числовым:


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [7]:
# Кол-во пропусков
print("Пропуски в каждой колонке:")
print(df.isnull().sum())

# Удаляем строки без CustomerID
df = df.dropna(subset=['CustomerID'])
print(f"После удаления NaN в CustomerID: {df.shape}")

# Проверяем пропуски в Description
# Удаляю, т.к. их немного
df = df.dropna(subset=['Description'])
print(f"После удаления NaN в Description: {df.shape}")

# Дубликаты
duplicates = df.duplicated().sum()
print(f"Дубликатов строк: {duplicates}")
df = df.drop_duplicates()
print(f"После удаления дубликатов: {df.shape}")

# Фильтр отрицательных или нулевых Quantity и UnitPrice
print("Количество строк с Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("Количество строк с UnitPrice <= 0:", (df['UnitPrice'] <= 0).sum())
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"После фильтрации положительных значений: {df.shape}")

# 1.4 Обработка отмен
df['IsCancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')
print("Отмененных транзакций:", df['IsCancelled'].sum())

# Для основного анализа будем использовать только неотмененные заказы
df_sales = df[~df['IsCancelled']].copy()
print(f"Таблица продаж (без отмен): {df_sales.shape}")

Пропуски в каждой колонке:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
IsCancelled    0
dtype: int64
После удаления NaN в CustomerID: (392692, 9)
После удаления NaN в Description: (392692, 9)
Дубликатов строк: 0
После удаления дубликатов: (392692, 9)
Количество строк с Quantity <= 0: 0
Количество строк с UnitPrice <= 0: 0
После фильтрации положительных значений: (392692, 9)
Отмененных транзакций: 0
Таблица продаж (без отмен): (392692, 9)


In [8]:
# Считаем общую выручку
df_sales['Revenue'] = df_sales['Quantity'] * df_sales['UnitPrice']

# Преобразовываем даты
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])
df_sales['Date'] = df_sales['InvoiceDate'].dt.date
df_sales['Year'] = df_sales['InvoiceDate'].dt.year
df_sales['Month'] = df_sales['InvoiceDate'].dt.month
df_sales['Day'] = df_sales['InvoiceDate'].dt.day
df_sales['DayOfWeek'] = df_sales['InvoiceDate'].dt.dayofweek  # 0=Monday, 6=Sunday
df_sales['Hour'] = df_sales['InvoiceDate'].dt.hour
df_sales['IsWeekend'] = df_sales['DayOfWeek'].isin([5, 6])  # суббота, воскресенье

# Агрегация до уровня заказа (Invoice)
invoice_agg = df_sales.groupby('InvoiceNo').agg({
    'CustomerID': 'first',
    'InvoiceDate': 'first',
    'Revenue': 'sum',
    'Quantity': 'sum'
}).reset_index()
invoice_agg['Date'] = invoice_agg['InvoiceDate'].dt.date
print("Количество уникальных заказов:", invoice_agg.shape[0])

Количество уникальных заказов: 18532


In [9]:
snapshot_date = df_sales['InvoiceDate'].max() + timedelta(days=1)
print("Дата отсчета для RFM:", snapshot_date)

# Группировка по клиенту
rfm = df_sales.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',  # Frequency – количество уникальных заказов
    'Revenue': 'sum'         # Monetary
}).rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'Revenue': 'Monetary'})

print("RFM таблица (первые 5):")
display(rfm.head())

# Присваиваем баллы (как квартили)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])  # 4 – самые недавние
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4])

# 3.4 Общий RFM Score
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# 3.5 Сегментация по правилам
def rfm_segment(row):
    if row['R_Score'] == 4 and row['F_Score'] == 4 and row['M_Score'] == 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] >= 3 and row['F_Score'] <= 2:
        return 'Promising'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost'
    else:
        return 'Other'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)

print("Распределение по сегментам:")
print(rfm['Segment'].value_counts())

# dim_customer
dim_customer = rfm[['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment']].reset_index()
print("dim_customer готов, размер:", dim_customer.shape)

Дата отсчета для RFM: 2011-12-10 12:50:00
RFM таблица (первые 5):


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


Распределение по сегментам:
Segment
Lost         1504
Loyal        1034
Promising     665
At Risk       646
Champions     489
Name: count, dtype: int64
dim_customer готов, размер: (4338, 9)


In [10]:
# очищенные транзакции с нужными колонками
fact_sales = df_sales[['InvoiceNo', 'StockCode', 'CustomerID', 'Date', 'Quantity', 'UnitPrice', 'Revenue']].copy()
# Для связи с dim_customer убедимся, что CustomerID есть в dim_customer
fact_sales = fact_sales[fact_sales['CustomerID'].isin(dim_customer['CustomerID'])]
print("fact_sales размер:", fact_sales.shape)

# Уникальные товары
dim_product = df_sales[['StockCode', 'Description']].drop_duplicates().reset_index(drop=True)
print("dim_product размер:", dim_product.shape)

# Непрерывный календарь от минимальной даты до максимальной
min_date = fact_sales['Date'].min()
max_date = fact_sales['Date'].max()
date_range = pd.date_range(start=min_date, end=max_date, freq='D')
dim_date = pd.DataFrame({'Date': date_range.date})
dim_date['Year'] = date_range.year
dim_date['Month'] = date_range.month
dim_date['Day'] = date_range.day
dim_date['DayOfWeek'] = date_range.dayofweek
dim_date['IsWeekend'] = dim_date['DayOfWeek'].isin([5, 6])
dim_date['Quarter'] = date_range.quarter
dim_date['Month_Name'] = date_range.strftime('%B')
print("dim_date размер:", dim_date.shape)

# Сохранение CSV
fact_sales.to_csv('fact_sales.csv', index=False)
dim_customer.to_csv('dim_customer.csv', index=False)
dim_product.to_csv('dim_product.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)

print("Все таблицы сохранены успешно!")

fact_sales размер: (392692, 7)
dim_product размер: (3897, 2)
dim_date размер: (374, 8)
Все таблицы сохранены успешно!


In [11]:
print("Файлы сохранены:")
print(" - fact_sales.csv (факты продаж, строк: {})".format(fact_sales.shape[0]))
print(" - dim_customer.csv (клиенты с RFM, строк: {})".format(dim_customer.shape[0]))
print(" - dim_product.csv (товары, строк: {})".format(dim_product.shape[0]))
print(" - dim_date.csv (календарь, строк: {})".format(dim_date.shape[0]))

Файлы сохранены:
 - fact_sales.csv (факты продаж, строк: 392692)
 - dim_customer.csv (клиенты с RFM, строк: 4338)
 - dim_product.csv (товары, строк: 3897)
 - dim_date.csv (календарь, строк: 374)
